# Example 05 — Inverse design (Ax)

Two continuous KPIs (`permeance`, `rejection`). After training, **Ax** searches recipe-like numbers.

**Inverse design** = we know the performance we want and search for inputs. The optimizer only queries the **model**, not the lab.

This example uses a **scalarized** (weighted) score so it finishes quickly:

`0.3 * permeance + 0.7 * rejection` (after scaling). Changing 0.3 / 0.7 changes what “best” means.

The **full tutorial** uses **true multiobjective / Pareto** search: a *set* of recipes along the trade-off, not one winner. You can switch this example to that mode (below).

Needs `pip install -e ".[opt]"` (Colab bootstrap does this). First install can take several minutes.

```bash
python -m run_pipeline --config examples/05_inverse_design_controls/config.yaml --force-retrain
```

Suggestions are **not** a lab-proven membrane. `n_trials` is small on purpose.

[Open in Colab](https://colab.research.google.com/github/Oikiy/ML-MEM-Review/blob/main/examples/05_inverse_design_controls/Colab.ipynb)


## 1. Bootstrap (Colab clone + install)

**What you will do.** Run the next cell so this notebook can see the full `ML-MEM-Review` **repository** (`config.yaml`, `src/`, data) — not just this `.ipynb`.

**On Colab** it clones or updates `/content/ML-MEM-Review` and pip-installs the package. First install can take a few minutes. No Google Drive.

**Locally** it finds the repo (example 05 will install Ax if missing).

**What to look for.** `Repo root: ...` and `Colab bootstrap done.`

**Common pitfall.** Running later cells first. Expand the hidden **Bootstrap** form if clone/install fails. Run cells **in order** after this.


In [ ]:
# @title Bootstrap (clone + install)
# === COLAB BOOTSTRAP (safe locally) =========================================
import os
import sys
import subprocess
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, **kwargs):
    print("+", " ".join(cmd) if isinstance(cmd, (list, tuple)) else cmd)
    subprocess.check_call(cmd, **kwargs)

def _find_repo_root(start: Path):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

IN_COLAB = _in_colab()
REPO_URL = os.environ.get("ML_MEM_REPO_URL", "https://github.com/Oikiy/ML-MEM-Review.git")
REPO_NAME = os.environ.get("ML_MEM_REPO_NAME", "ML-MEM-Review")
REPO_REF = os.environ.get("ML_MEM_REPO_REF", "main")
COLAB_ROOT = Path(os.environ.get("ML_MEM_COLAB_ROOT", f"/content/{REPO_NAME}"))
EXTRAS = 'opt'

def _git_head(repo: Path) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError, OSError):
        return ""

def _truthy(name: str) -> bool:
    return os.environ.get(name, "").strip().lower() in {"1", "true", "yes"}

if IN_COLAB:
    try:
        if COLAB_ROOT.exists() and (COLAB_ROOT / ".git").is_dir():
            if _truthy("ML_MEM_SKIP_PULL"):
                print(f"Skipping git update at {COLAB_ROOT} (ML_MEM_SKIP_PULL=1)")
            else:
                print(f"Updating clone at {COLAB_ROOT} to origin/{REPO_REF}")
                _run(["git", "-C", str(COLAB_ROOT), "fetch", "--depth", "1", "origin", REPO_REF])
                _run(["git", "-C", str(COLAB_ROOT), "reset", "--hard", "FETCH_HEAD"])
        elif not COLAB_ROOT.exists():
            _run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)])
        os.chdir(COLAB_ROOT)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "git clone/update failed. Make the repo public, set ML_MEM_REPO_URL "
            f"to a fork, or set ML_MEM_REPO_REF. Original error: {exc}"
        ) from exc
    root = _find_repo_root(Path.cwd())
else:
    root = _find_repo_root(Path.cwd())

if root is None:
    raise RuntimeError(
        "Could not locate ML-MEM-Review (need config.yaml + src/). "
        "On Colab: re-run this cell after a successful clone. "
        "Locally: open from inside a full clone of the repository."
    )

os.chdir(root)
print(f"Repo root: {root}  git={_git_head(root)[:12]}")

src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB:
    marker = root / ".colab_install_ok"
    force = _truthy("ML_MEM_FORCE_INSTALL")
    head = _git_head(root)
    extras = EXTRAS
    pip_spec = str(root) if not extras else f"{root}[{extras}]"
    install_key = f"{head}\n{extras}"
    prev = marker.read_text(encoding="utf-8").strip() if marker.is_file() else ""
    if force or prev != install_key:
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", pip_spec])
        try:
            marker.write_text(install_key + "\n", encoding="utf-8")
        except OSError:
            pass
    else:
        print("Skipping pip install (clone SHA + extras unchanged). Set ML_MEM_FORCE_INSTALL=1 to reinstall.")
    print("Colab bootstrap done.")
else:
    print("Local session: repo root + src/ on path (no auto pip install).")

if EXTRAS:
    try:
        import ax  # noqa: F401
    except ImportError:
        pip_spec = f"{root}[{EXTRAS}]"
        print(f"Ax not found in this kernel; installing {pip_spec}")
        _run([sys.executable, "-m", "pip", "install", "-e", pip_spec])

REPO_ROOT = root
# === end bootstrap ===========================================================


## What this config emphasizes

- **`search_space`:** numbers Ax may change, plus a choice (`monomer`).
- **`constant_parameters.solvent: hexane`:** still a model input, not searched.
- **`mode: scalarized`** and the 0.3 / 0.7 equation — edit the weights.
- **True multiobjective (Pareto):** set `mode: multiobjective` and uncomment each objective’s `threshold`. Response maps then show **blue triangles** (Pareto) vs red circles (all trials). That is what the membrane tutorial uses.
- Small `n_trials`. Two targets ⇒ two models before search.

**Common pitfall.** Missing Ax → `--stage optimize` **errors** (it does not skip). Searchable + constant names must match training features. After changing `mode`, delete `models/ax_client.pkl` or use `--force-retrain`.


## 2. Read `config.yaml`

**What you will do.** Print this example’s YAML. Nothing is trained yet.

**Why.** This gallery teaches *configuration*. The keys you change here are how you reuse `run_pipeline` on your own table without rewriting Python.

**What to look for.** The settings listed under “What this config emphasizes.”

**Try it yourself.** Before running the pipeline, name the **target** column you expect the model to predict.


In [ ]:
# Show the example config (edit the file under examples/ to experiment)
from pathlib import Path

CONFIG_REL = "examples/05_inverse_design_controls/config.yaml"
CONFIG_PATH = REPO_ROOT / CONFIG_REL
assert CONFIG_PATH.is_file(), f"Missing {CONFIG_PATH} — bootstrap/clone incomplete?"

print(CONFIG_PATH.read_text(encoding="utf-8"))



## 3. Run the pipeline

**What you will do.** The next cell calls `python -m run_pipeline` for this folder, **one stage at a time** (EDA, then model, then optimize if this is example 05).

That is the same command you would run in a terminal. Results are written under `examples/05_inverse_design_controls/`, not into the notebook file.

**What to look for.** Printed `Pipeline finished`. On Colab, open `examples/05_inverse_design_controls/outputs/figures` in the left file browser.

**Common pitfall.** The red `RuntimeError` at the bottom is only a wrapper. Read the traceback **above** it. `--force-retrain` rebuilds models after you edit YAML.


In [ ]:
# Run the pipeline (artifacts written under examples/05_inverse_design_controls/)
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.copy()
src = str(Path(REPO_ROOT) / "src")
env["PYTHONPATH"] = src + os.pathsep + env.get("PYTHONPATH", "")
try:
    import ax  # noqa: F401
except ImportError:
    spec = f"{REPO_ROOT}[opt]"
    print(f"Ax not found; installing {spec}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
base = [sys.executable, "-m", "run_pipeline", "--config", str(CONFIG_PATH), "--force-retrain"]
for stage in ("eda", "model", "optimize"):
    cmd = base + ["--stage", stage]
    print("+", " ".join(cmd))
    completed = subprocess.run(
        cmd, cwd=str(REPO_ROOT), env=env, text=True, capture_output=True
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        err = (completed.stderr or completed.stdout or "").strip()
        raise RuntimeError(
            f"run_pipeline --stage {stage} exited {completed.returncode}.\n\n{err[-8000:]}"
        )
print("Done.")



## 4. Show figures

The next cell opens the PNGs in **workflow order**. Missing `inverse_design_summary.png` means optimize did not run (install Ax / re-run bootstrap).

### EDA and two parity plots
Same language as example 01, twice: one parity for `permeance`, one for `rejection`. **Diagonal** = prediction matches measurement. Watch **test** points, not only train. These two models are what Ax will query next (the **oracle** = “ask the saved models”).

### Inverse-design summary (3D “waterfall”)
Each ribbon is one quantity vs **trial index** (0, 1, 2, … as Ax tries recipes). You want to see the combined score (or KPIs) **move**, not a flat line after the random start. This example is **scalarized**: one weighted score (`0.3` permeance + `0.7` rejection), so there is a **single** notion of “best,” not a Pareto set.

### Response maps (contour then surface)
A **response map** is a smoothed landscape of a score vs two knobs, interpolated from the **few trials** you ran — a cartoon, not a map of nature.

- **Contour** = 2D (easier to read). **Surface** = 3D view of the **same** interpolation.
- **Red circles** = all Ax trials.
- **Blue triangles** = **Pareto** points (not dominated on both KPIs). They appear only if you set `mode: multiobjective`. In the default scalarized example you should **not** expect triangles.

**Try it yourself.** If you set the rejection weight to 1.0 and permeance to 0.0, should suggested recipes chase rejection or a balance of both?


In [ ]:
# @title Show figures
from pathlib import Path
from IPython.display import display, Markdown, Image

fig_dir = REPO_ROOT / "examples" / "05_inverse_design_controls" / "outputs" / "figures"
summary = fig_dir / "inverse_design_summary.png"
if not summary.is_file():
    raise FileNotFoundError(
        f"Missing {summary}. Inverse design did not run. "
        "Install Ax (`pip install -e '.[opt]'`) and re-run bootstrap."
    )
for name in [
    "target_distributions.png",
    "pearson_heatmap.png",
    "parity_permeance.png",
    "parity_rejection.png",
    "inverse_design_summary.png",
    "response_map_1_contour.png",
    "response_map_1_surface.png",
    "response_map_2_contour.png",
    "response_map_2_surface.png",
]:
    path = fig_dir / name
    if path.is_file():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))


## Try next

- Change the 0.3 / 0.7 weights and re-run the pipeline cell.
- Switch to Pareto: `mode: multiobjective` and uncomment `threshold` on each objective, then re-run optimize.
- Tighten `temperature_C` bounds.
- Full tutorial §6: [`Ml-MEM-Tut.ipynb`](../../notebooks/Complete_notebook/Ml-MEM-Tut.ipynb)
